# Imports

In [ ]:
import random
import requests
import folium
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
from folium.plugins import HeatMap
from shapely.geometry import Point
from geopy.distance import geodesic
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA

In [ ]:
OD_PAIRS_COUNT = 20
ROUTES_PER_OD = 100

# Generating the Dataset of Origin-Destination Pairs

Offical government data for the exact bounds of Vienna:

In [ ]:
vienna = gpd.read_file("resources/vienna.json")
vienna_polygon = vienna.geometry.union_all()

minx, miny, maxx, maxy = vienna.total_bounds

print(f"West:  {minx}")
print(f"South: {miny}")
print(f"East:  {maxx}")
print(f"North: {maxy}")

center = [
    (miny + maxy) / 2,
    (minx + maxx) / 2
]

m = folium.Map(
    location=center,
    zoom_start=11
)

folium.GeoJson(vienna.geometry.to_json()).add_to(m)

In [ ]:
def sample_point_in_polygon(polygon):
    minx, miny, maxx, maxy = polygon.bounds

    while True:
        p = Point(
            random.uniform(minx, maxx),
            random.uniform(miny, maxy)
        )
        if polygon.contains(p):
            return p


def sample_od_pair(polygon):
    start = sample_point_in_polygon(polygon)

    lat, lon = start.y, start.x

    distance_km = random.uniform(2, 10)
    bearing = random.uniform(0, 360)

    dest = geodesic(kilometers=distance_km).destination((lat, lon), bearing)

    end = Point(dest.longitude, dest.latitude)

    if polygon.contains(end):
        return start, end
    else:
        return sample_od_pair(polygon)

In [ ]:
pairs = []

while len(pairs) < OD_PAIRS_COUNT:
    pair = sample_od_pair(vienna_polygon)
    pairs.append(pair)

In [ ]:
for i, (start, end) in enumerate(pairs):

    # Startpunkt
    folium.CircleMarker(
        location=[start.y, start.x],
        radius=3,
        color="green",
        fill=True,
        fill_opacity=0.8,
        popup=f"start {i}"
    ).add_to(m)

    # Endpunkt
    folium.CircleMarker(
        location=[end.y, end.x],
        radius=3,
        color="red",
        fill=True,
        fill_opacity=0.8,
        popup=f"end {i}"
    ).add_to(m)

    # Verbindungslinie
    folium.PolyLine(
        locations=[
            [start.y, start.x],
            [end.y, end.x]
        ],
        color="blue",
        weight=2,
        opacity=0.5
    ).add_to(m)

m

# Configuration

In [ ]:
API_URL = "http://localhost:8080/api/route"

START = (48.18051, 16.33375)
END = (48.23336, 16.37512)

# API Request

In [ ]:
def get_route(start, end, preferences):

    payload = {
        "points": [
            {
                "lat": start[0],
                "lon": start[1]
            },
            {
                "lat": end[0],
                "lon": end[1]
            }
        ],
        "profile": "BASE",
        "mode": "CUSTOM",
        "withInstructions": False,
        "preferencesDto": preferences
    }

    response = requests.post(
        API_URL,
        json=payload
    )

    response.raise_for_status()

    return response.json()

# Parameters and Test Profiles

In [ ]:
def route_params(
        cycleway_lane=1.0,
        cycleway_track=1.0,
        road_class_cycleway=1.0,
        road_class_primary_secondary_trunk=1.0,
        road_class_residential=1.0,
        road_class_path=1.0,
        road_class_footway=1.0,
        surface_cobblestone_gravel_unpaved=1.0,
        incline_avg_above_four_percent=1.0,
        incline_max_above_four_percent=1.0,
        decline_avg_above_four_percent=1.0,
        decline_max_above_four_percent=1.0,
        no_car_access=1.0,
        bike_road_access_designated=1.0,
        bike_road_access_dismount_or_get_off_bike=1.0,
        max_speed_above_thirty=1.0):
    
    return {
        "cyclewayLane": cycleway_lane,
        "cyclewayTrack": cycleway_track,
        "roadClassCycleway": road_class_cycleway,
        "roadClassPrimarySecondaryTrunk": road_class_primary_secondary_trunk,
        "roadClassResidential": road_class_residential,
        "roadClassPath": road_class_path,
        "roadClassFootway": road_class_footway,
        "surfaceCobblestoneGravelUnpaved": surface_cobblestone_gravel_unpaved,
        "inclineAvgAboveFourPercent": incline_avg_above_four_percent,
        "inclineMaxAboveFourPercent": incline_max_above_four_percent,
        "declineAvgAboveFourPercent": decline_avg_above_four_percent,
        "declineMaxAboveFourPercent": decline_max_above_four_percent,
        "noCarAccess": no_car_access,
        "bikeRoadAccessDesignated": bike_road_access_designated,
        "bikeRoadAccessDismountOrGetOffBike": bike_road_access_dismount_or_get_off_bike,
        "maxSpeedAboveThirty": max_speed_above_thirty,
    }


BASE_PREFS = route_params()

SAFE_PREFS = route_params(
    road_class_cycleway=1.5,
    #no_car_access=1.5,
    #road_class_primary_secondary_trunk=0.5,
    #max_speed_above_thirty=0.5
)

# Send Requests

In [ ]:
route_base = get_route(
    START,
    END,
    BASE_PREFS
)

route_safe = get_route(
    START,
    END,
    SAFE_PREFS
)

# Visualize on Map

In [ ]:
def plot_routes(route_a, route_b):

    coords_a = [
        (lat, lon)
        for lon, lat in route_a["geometry"]["coordinates"]
    ]

    coords_b = [
        (lat, lon)
        for lon, lat in route_b["geometry"]["coordinates"]
    ]

    center = coords_a[0]

    m = folium.Map(
        location=center,
        zoom_start=14
    )

    folium.PolyLine(
        coords_a,
        color="blue",
        weight=6,
        tooltip="BASE"
    ).add_to(m)

    folium.PolyLine(
        coords_b,
        color="red",
        weight=4,
        tooltip="SAFE"
    ).add_to(m)

    return m

plot_routes(
    route_base,
    route_safe
)

# Extract Feature Vector

In [ ]:
def extract_feature_vector(route):

    vector = {}

    features = route["properties"]["routeFeatures"]

    for category, values in features.items():

        for value, share in values.items():

            vector[f"{category}:{value}"] = share

    return vector

In [ ]:
v_base = extract_feature_vector(route_base)
v_safe = extract_feature_vector(route_safe)

df = pd.DataFrame({
    "BASE": v_base,
    "SAFE": v_safe
}).fillna(0)

df

# Plot as Bar Chart

In [ ]:
df.plot(
    kind="bar",
    figsize=(16,5)
)

plt.ylabel("Share")
plt.title("Route Feature Comparison")
plt.tight_layout()
plt.show()

# Absolute Differences

In [ ]:
difference = abs(
    df["BASE"] - df["SAFE"]
)

difference.sort_values(
    ascending=False
)

difference.sort_values().plot.barh(
    figsize=(8,6)
)

plt.title(
    "Absolute Feature Differences"
)

plt.show()

# L1 Distance

In [ ]:
def l1_distance(v1, v2):

    keys = set(v1.keys()) | set(v2.keys())

    return sum(
        abs(
            v1.get(k, 0)
            - v2.get(k, 0)
        )
        for k in keys
    )

distance = l1_distance(
    v_base,
    v_safe
)

print(
    f"L1 Distance: {distance:.4f}"
)

# Try out random weights

In [ ]:
def sample_theta():
    THETA_MIN = 0.2
    THETA_MAX = 2.0

    return route_params(
        cycleway_lane=random.uniform(THETA_MIN, THETA_MAX),
        cycleway_track=random.uniform(THETA_MIN, THETA_MAX),
        road_class_cycleway=random.uniform(THETA_MIN, THETA_MAX),
        road_class_primary_secondary_trunk=random.uniform(THETA_MIN, THETA_MAX),
        road_class_residential=random.uniform(THETA_MIN, THETA_MAX),
        road_class_path=random.uniform(THETA_MIN, THETA_MAX),
        road_class_footway=random.uniform(THETA_MIN, THETA_MAX),
        surface_cobblestone_gravel_unpaved=random.uniform(THETA_MIN, THETA_MAX),
        incline_avg_above_four_percent=random.uniform(THETA_MIN, THETA_MAX),
        incline_max_above_four_percent=random.uniform(THETA_MIN, THETA_MAX),
        decline_avg_above_four_percent=random.uniform(THETA_MIN, THETA_MAX),
        decline_max_above_four_percent=random.uniform(THETA_MIN, THETA_MAX),
        no_car_access=random.uniform(THETA_MIN, THETA_MAX),
        bike_road_access_designated=random.uniform(THETA_MIN, THETA_MAX),
        bike_road_access_dismount_or_get_off_bike=random.uniform(THETA_MIN, THETA_MAX),
        max_speed_above_thirty=random.uniform(THETA_MIN, THETA_MAX),
    )

In [ ]:
# Random Exploration Loop

In [ ]:
def run_od_experiment(start, end, theta):
    route = get_route(start, end, theta)
    vector = extract_feature_vector(route)

    return route, vector

In [ ]:
rows = []
all_routes = []

for i, (start_p, end_p) in enumerate(pairs):

    start = (start_p.y, start_p.x)
    end = (end_p.y, end_p.x)

    try:
        base_route = get_route(start, end, BASE_PREFS)
        v_base_local = extract_feature_vector(base_route)

        all_routes.append(base_route)

        for j in range(ROUTES_PER_OD):

            theta = sample_theta()

            route, vector = run_od_experiment(start, end, theta)

            dist = l1_distance(v_base_local, vector)

            feature_row = {
                "od_id": i,
                "sample_id": j,
                "route_length": route["properties"]["distance"],
                "route_time": route["properties"]["time"],
                **theta,
                "distance": dist
            }

            for k, v in vector.items():
                feature_row[k] = v
            
            rows.append(feature_row)

            all_routes.append(route)

    except Exception as e:
        print(f"failed OD {i}: {e}")

# Show Generated Routes

In [ ]:
def plot_route_overlay(routes):

    import folium

    m = folium.Map(location=[48.21, 16.34], zoom_start=13)

    colors = ["blue", "red", "green", "purple", "orange"]

    for i, r in enumerate(routes):

        coords = [
            (lat, lon)
            for lon, lat in r["geometry"]["coordinates"]
        ]

        folium.PolyLine(
            coords,
            color=colors[i % len(colors)],
            weight=3,
            opacity=0.7
        ).add_to(m)

    return m

plot_route_overlay(all_routes)

In [ ]:
# Evaluations

In [ ]:
results_df = pd.DataFrame(rows)
results_df["distance"].describe()
results_df.groupby("od_id").mean(numeric_only=True)
results_df.groupby("od_id").std(numeric_only=True)
results_df.groupby("od_id").agg(
    ["min", "max", "mean", "std"]
)

In [ ]:
summary = (
    results_df
    .groupby("od_id")
    .agg({
        "route_length": ["min", "max", "mean", "std"],
        "route_time": ["min", "max", "mean", "std"],
        "distance": ["min", "max", "mean", "std"]
    })
)

summary

In [ ]:
plt.figure(figsize=(8,4))

plt.hist(
    results_df["distance"],
    bins=30
)

plt.xlabel("L1 Distance")
plt.ylabel("Count")
plt.title("Distribution of Route Feature Distances")

plt.show()

In [ ]:
feature_cols = [
    c for c in results_df.columns
    if ":" in c
]

variation = (
    results_df[feature_cols]
    .std()
    .sort_values(ascending=False)
)

variation

plt.figure(figsize=(10,8))

variation.plot.barh()

plt.xlabel("Standard Deviation")
plt.title("Global Feature Variability")

plt.show()

In [ ]:
plt.figure(figsize=(10,5))

results_df.boxplot(
    column="route_length",
    by="od_id"
)

plt.ylabel("Meters")
plt.title("Route Length Distribution")
plt.suptitle("")

plt.show()

In [ ]:
results_df.boxplot(
    column="route_time",
    by="od_id"
)

In [ ]:
results_df.boxplot(
    column="cycleway:lane",
    by="od_id"
)

In [ ]:
feature_cols = [
    c for c in results_df.columns
    if ":" in c
]

variation_matrix = (
    results_df
    .groupby("od_id")[feature_cols]
    .std()
)

plt.figure(figsize=(16,8))

sns.heatmap(
    variation_matrix,
    cmap="viridis"
)

plt.title(
    "Feature Variability Across OD Pairs"
)

plt.show()

In [ ]:
diversity = (
    results_df
    .groupby("od_id")[feature_cols]
    .std()
    .mean(axis=1)
)

plt.figure(figsize=(8,4))

diversity.sort_values().plot.bar()

plt.ylabel("Mean Feature Std")
plt.xlabel("OD Pair")
plt.title("Route Diversity Score per OD Pair")

plt.show()

In [ ]:
length_stats = (
    results_df
    .groupby("od_id")["route_length"]
    .agg(["mean", "std"])
)

length_stats["cv"] = (
    length_stats["std"]
    / length_stats["mean"]
)

length_stats["cv"].plot.bar()

plt.ylabel("Coefficient of Variation")
plt.title("Relative Route Length Variability")

plt.show()

## How much do the routes differ?

In [ ]:
behavior_space = pd.DataFrame({
    "min": results_df[feature_cols].min(),
    "max": results_df[feature_cols].max(),
    "mean": results_df[feature_cols].mean(),
    "std": results_df[feature_cols].std()
})

behavior_space["range"] = (
    behavior_space["max"]
    - behavior_space["min"]
)

behavior_space.sort_values(
    "range",
    ascending=False
)

behavior_space["range"] \
    .sort_values() \
    .plot.barh(figsize=(10,8))

plt.xlabel("Reachable Range")
plt.title("Behavior Space of Route Features")

plt.show()

## Clustering

In [ ]:
X = results_df[feature_cols].fillna(0)

X_scaled = StandardScaler().fit_transform(X)

kmeans = KMeans(
    n_clusters=4,
    random_state=42
)

results_df["cluster"] = kmeans.fit_predict(X_scaled)

results_df["cluster"].value_counts()

cluster_profiles = (
    results_df
    .groupby("cluster")[feature_cols]
    .mean()
)

cluster_profiles

In [ ]:
pca = PCA(n_components=2)

X_pca = pca.fit_transform(X_scaled)

plt.figure(figsize=(8,6))

plt.scatter(
    X_pca[:,0],
    X_pca[:,1],
    c=results_df["cluster"]
)

plt.xlabel("PC1")
plt.ylabel("PC2")

plt.show()

## Parameter Sensitivity

In [ ]:
results_df["track_bin"] = pd.cut(
    results_df["cyclewayTrack"],
    bins=5
)

response = (
    results_df
    .groupby("track_bin")
    ["cycleway:lane"]
    .mean()
)

response.plot(
    marker="o"
)

plt.ylabel(
    "Mean cycleway:lane share"
)

plt.xlabel(
    "cyclewayTrack bin"
)

plt.show()

# Feature ranges

## For a single OD pair

In [ ]:
od = 3

od_df = results_df[
    results_df["od_id"] == od
]

feature_cols = [
    c for c in od_df.columns
    if ":" in c
]

base_route = get_route(
    start,
    end,
    BASE_PREFS
)

v_base = extract_feature_vector(base_route)

behavior_space = pd.DataFrame({
    "min_share": od_df[feature_cols].min(),
    "max_share": od_df[feature_cols].max(),
})

behavior_space["base_share"] = [
    v_base.get(feature, 0)
    for feature in behavior_space.index
]

behavior_space["range"] = (
    behavior_space["max_share"]
    - behavior_space["min_share"]
)

behavior_space = (
    behavior_space
    .sort_values("range", ascending=False)
)

behavior_space

In [ ]:
top = behavior_space.head(15)

plt.figure(figsize=(10,8))

plt.hlines(
    y=top.index,
    xmin=top["min_share"],
    xmax=top["max_share"]
)

plt.scatter(
    top["base_share"],
    top.index,
    marker="o"
)

plt.xlabel("Feature Share")
plt.title(f"Reachable Behavior Space (OD {od})")

plt.show()

## For all OD pairs

In [ ]:
for od_id in results_df["od_id"].unique():

    od_df = results_df[
        results_df["od_id"] == od_id
    ]

    # Base-Route dieses OD
    base_vector = base_vectors[od_id]

    for feature in feature_cols:

        min_share = od_df[feature].min()
        max_share = od_df[feature].max()
        base_share = base_vector.get(feature, 0)

        rows.append({
            "od_id": od_id,
            "feature": feature,
            "min_share": min_share,
            "base_share": base_share,
            "max_share": max_share,
            "range": max_share - min_share,
            "delta_minus": base_share - min_share,
            "delta_plus": max_share - base_share
        })

behavior_df = pd.DataFrame(rows)